# 03 — Modeling Baselines and Model Comparison

Project: **Predicting Building Construction Era from French DPE data**

This notebook implements the first modeling pipeline for the project:

1. Load the cleaned dataset produced by the cleaning/EDA notebook.
2. Prepare features and target labels.
3. Train multiple classification models.
4. Evaluate them with accuracy, macro-F1, classification report, and confusion matrix.
5. Run a simple feature ablation study.

The target is expected to be a construction-era class such as:

- `pre_1948`
- `1948_1974`
- `1975_1988`
- `1989_2000`
- `2001_2012`
- `2013_plus`

If your cleaned file uses different column names, edit the configuration cell below.


## Method summary

We compare several methods:

### 1. Majority-class baseline
Always predicts the most frequent construction era.  
This is the minimum benchmark: every real model should beat it.

### 2. Logistic Regression
A linear classifier.  
It is useful because it is simple, fast, and interpretable.

### 3. Random Forest
An ensemble of decision trees trained on bootstrapped samples.  
It captures nonlinear relationships and feature interactions.

### 4. Gradient Boosting
A tree-based ensemble where trees are added sequentially to correct previous errors.  
It often performs well on tabular datasets.

### Evaluation metrics

We report:

- **Accuracy**: global percentage of correct predictions.
- **Macro-F1**: average F1 across classes, treating all eras equally.
- **Precision/recall per class**: useful when some eras are harder to predict.
- **Confusion matrix**: shows which eras the model confuses.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn utilities
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42


## 1. Configuration

Edit these values if your cleaned dataset uses different names.

The notebook assumes that Person A's cleaned dataset contains:

- one target column for construction era,
- several numerical features,
- several categorical features.


In [ ]:
# Path to the cleaned dataset produced by the cleaning/EDA notebook.
# Change this if your file has a different name or location.
DATA_PATH = "../data/clean/dpe_clean_model_ready.csv"

# Target column.
# Change this to the exact name in your cleaned dataset.
TARGET_COL = "construction_era"

# Optional: columns to drop before modeling.
# Keep identifiers, raw addresses, free text columns, or leakage columns out of the model.
DROP_COLS = [
    "construction_year",   # drop if target era was created from this
    "id",
    "address",
    "adresse",
    "numero_dpe"
]


## 2. Load cleaned dataset

If this cell fails, check:

1. the dataset path,
2. the target column name,
3. whether the cleaning notebook has already exported the clean CSV.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df[TARGET_COL].value_counts(dropna=False))


## 3. Basic dataset checks

We remove rows with missing target labels and drop obvious non-feature columns if they exist.


In [ ]:
# Remove rows without target
df = df.dropna(subset=[TARGET_COL]).copy()

# Drop configured columns only if they exist
existing_drop_cols = [col for col in DROP_COLS if col in df.columns]
df = df.drop(columns=existing_drop_cols)

print("Dropped columns:", existing_drop_cols)
print("Remaining shape:", df.shape)

# Separate features and target
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(str)

print("Feature matrix:", X.shape)
print("Target:", y.shape)


## 4. Identify numerical and categorical features

Scikit-learn models need numerical arrays.  
We therefore:

- impute missing numerical values with the median,
- scale numerical values for logistic regression,
- impute categorical values with `"missing"`,
- one-hot encode categorical variables.


In [ ]:
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features[:20])

print("\nCategorical features:", len(categorical_features))
print(categorical_features[:20])


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


## 5. Train/test split

We use a stratified split so that every construction era is represented proportionally in both train and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTrain class distribution:")
display(y_train.value_counts(normalize=True).sort_index())

print("\nTest class distribution:")
display(y_test.value_counts(normalize=True).sort_index())


## 6. Define models

Each model is wrapped in a pipeline:

```text
raw dataframe
→ preprocessing
→ model
→ prediction
```

This prevents data leakage because preprocessing is fitted only on the training data.


In [ ]:
models = {
    "majority_baseline": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),

    "logistic_regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "random_forest": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "hist_gradient_boosting": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        ))
    ])
}


## 7. Evaluation function

This function trains a model and reports:

- accuracy,
- macro-F1,
- classification report,
- confusion matrix.


In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    print("=" * 80)
    print(f"Training: {name}")
    print("=" * 80)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    ax.set_title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": macro_f1
    }


## 8. Train and compare all models

In [ ]:
results = []

for name, model in models.items():
    result = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)

results_df = pd.DataFrame(results).sort_values(by="macro_f1", ascending=False)
display(results_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
results_df.set_index("model")[["accuracy", "macro_f1"]].plot(kind="bar", ax=ax)
ax.set_title("Model comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Stratified cross-validation

Cross-validation evaluates models more robustly than a single train/test split.

We use macro-F1 because the construction-era classes may be imbalanced.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []

for name, model in models.items():
    print(f"Cross-validating: {name}")

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=-1,
        return_train_score=False
    )

    cv_results.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_macro_f1_mean": scores["test_f1_macro"].mean(),
        "cv_macro_f1_std": scores["test_f1_macro"].std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="cv_macro_f1_mean", ascending=False)
display(cv_results_df)


## 10. Feature ablation study

The project asks for ablation comparisons:

1. energy-only features,
2. structural-only features,
3. all features combined.

Edit the keyword lists below based on the exact column names in your cleaned dataset.


In [ ]:
ENERGY_KEYWORDS = [
    "conso",
    "energie",
    "energy",
    "dpe",
    "ges",
    "ghg",
    "emission",
    "chauffage",
    "heating",
    "ecs",
    "eau_chaude"
]

STRUCTURAL_KEYWORDS = [
    "surface",
    "mur",
    "wall",
    "toiture",
    "roof",
    "plancher",
    "floor",
    "fenetre",
    "window",
    "vitrage",
    "glazing",
    "isolation",
    "type_batiment",
    "building"
]

def columns_matching_keywords(columns, keywords):
    selected = []
    for col in columns:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in keywords):
            selected.append(col)
    return selected

energy_features = columns_matching_keywords(X.columns, ENERGY_KEYWORDS)
structural_features = columns_matching_keywords(X.columns, STRUCTURAL_KEYWORDS)

print("Energy features:", len(energy_features))
print(energy_features)

print("\nStructural features:", len(structural_features))
print(structural_features)


In [ ]:
def make_preprocessor_for_columns(X_subset):
    num_cols = X_subset.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols = X_subset.select_dtypes(exclude=["number", "bool"]).columns.tolist()

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ],
        remainder="drop"
    )

def run_ablation(feature_sets, base_model):
    ablation_results = []

    for feature_set_name, cols in feature_sets.items():
        if len(cols) == 0:
            print(f"Skipping {feature_set_name}: no columns selected")
            continue

        X_sub = X[cols].copy()
        preproc = make_preprocessor_for_columns(X_sub)

        model = Pipeline(steps=[
            ("preprocess", preproc),
            ("model", base_model)
        ])

        scores = cross_validate(
            model,
            X_sub,
            y,
            cv=cv,
            scoring=["accuracy", "f1_macro"],
            n_jobs=-1
        )

        ablation_results.append({
            "feature_set": feature_set_name,
            "n_features": len(cols),
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_std": scores["test_accuracy"].std(),
            "macro_f1_mean": scores["test_f1_macro"].mean(),
            "macro_f1_std": scores["test_f1_macro"].std()
        })

    return pd.DataFrame(ablation_results).sort_values(by="macro_f1_mean", ascending=False)

feature_sets = {
    "energy_only": energy_features,
    "structural_only": structural_features,
    "all_features": X.columns.tolist()
}

ablation_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

ablation_df = run_ablation(feature_sets, ablation_model)
display(ablation_df)


## 11. Save results

These CSV files can be used later in the final report or presentation.


In [ ]:
import os

OUTPUT_DIR = "../reports/modeling"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_df.to_csv(f"{OUTPUT_DIR}/test_set_model_comparison.csv", index=False)
cv_results_df.to_csv(f"{OUTPUT_DIR}/cross_validation_results.csv", index=False)
ablation_df.to_csv(f"{OUTPUT_DIR}/feature_ablation_results.csv", index=False)

print("Saved results to:", OUTPUT_DIR)


## Conclusion template

Fill this in after running the notebook.

Example:

> The majority-class baseline obtained a macro-F1 of X. Logistic Regression improved over the baseline, showing that the selected DPE features contain useful signal. Random Forest and Gradient Boosting performed better, suggesting nonlinear relationships between building characteristics and construction era. The ablation study showed that [energy / structural / all] features were most informative. Confusion matrices indicate that adjacent eras are often confused, which is expected because renovation and gradual regulatory changes blur the boundaries between construction periods.
